# 500 — CRISPR Associations

## Objective

Evaluate computational associations between the three frozen Phase 4
cross-system transcriptomic consensus programs and DepMap Public 24Q4 CRISPR
gene-effect profiles in the overlapping cancer-cell-model cohort.

The Phase 4 consensus-program identities, orientations, gene weights,
source-system mappings, and cell-line consensus scores are frozen upstream
inputs. Notebook 500 does not refit, reweight, redefine, rescue, exclude, or
rename any consensus representation on the basis of CRISPR results.

## Analytical status

Notebook 500 is a prespecified downstream functional-vulnerability
characterization over a frozen program universe. The gene-level dependency
screen is hypothesis-generating: observed associations may provide evidence
for putative functional vulnerabilities, but they do not establish causal
dependencies, validated targets, therapeutic targets, clinical biomarkers, or
mechanistic proof.

## Lineage-aware analytical boundary

The frozen cell-line consensus representations contain substantial lineage
structure. Pooled pan-cancer associations are therefore descriptive only.
Primary inference will be lineage-adjusted, while within-lineage analyses will
characterize effect consistency, heterogeneity, and lineage-restricted
associations when sample support is sufficient.

The frozen Phase 4 cell-line cohort defines the anchor model universe. CRISPR
coverage may reduce the analyzable subset through model overlap or gene-level
missingness, but models outside the frozen consensus cohort will not be
introduced downstream.

## CRISPR identifiers and analysis-ready representation

Original DepMap CRISPR gene labels will be preserved as analytical identifiers.
Gene symbols and Entrez IDs will be parsed transparently from the source
`SYMBOL (EntrezID)` labels without external HGNC remapping, alias rescue, or
silent duplicate collapse.

Model overlap, gene coverage, missingness, identifier parsing, and gene-level
informativeness will be characterized before the program–dependency
association family is frozen.

## Confounding, multiplicity, and interpretation

Lineage is a mandatory adjustment variable. Additional covariates will be used
only when a frozen, provenance-supported representation exists for the relevant
models; no proliferation or technical proxy will be constructed post hoc to
remove an unresolved limitation.

Gene-eligibility rules, lineage-support rules, the primary gene × program
hypothesis family, and multiple-testing correction will be frozen before
association results are inspected. Global descriptive associations,
lineage-adjusted inference, within-lineage evidence, and heterogeneity analyses
will remain explicitly separated by evidentiary role.

Negative, heterogeneous, lineage-specific, or non-recoverable results are
valid outputs. Association does not establish causality.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json
import re

import numpy as np
import pandas as pd
import statsmodels.api as sm

from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Input paths
# =============================================================================

crispr_gene_effect_path = Paths.depmap / "CRISPRGeneEffect.csv"

consensus_cellline_scores_path = (
    Paths.consensus_programs
    / "401_consensus_cellline_scores.parquet"
)

cell_line_cohort_path = (
    Paths.metadata
    / "302_integrated_modeling_cohort.csv"
)

In [3]:
# =============================================================================
# Load direct inputs
# =============================================================================

consensus_cellline_scores = pd.read_parquet(
    consensus_cellline_scores_path
)

crispr_gene_effect = pd.read_csv(
    crispr_gene_effect_path
)

cell_line_cohort = pd.read_csv(
    cell_line_cohort_path
)

In [4]:
# =============================================================================
# Inspect loaded input structure
# =============================================================================

print("Consensus cell-line scores:", consensus_cellline_scores.shape)
print("Consensus columns:", consensus_cellline_scores.columns.tolist())

print("\nCRISPR gene effect:", crispr_gene_effect.shape)
print("CRISPR first columns:", crispr_gene_effect.columns[:5].tolist())

Consensus cell-line scores: (713, 4)
Consensus columns: ['ModelID', 'CONSENSUS_TX_01', 'CONSENSUS_TX_02', 'CONSENSUS_TX_03']

CRISPR gene effect: (1178, 17917)
CRISPR first columns: ['Unnamed: 0', 'A1BG (1)', 'A1CF (29974)', 'A2M (2)', 'A2ML1 (144568)']


In [5]:
# =============================================================================
# Standardize CRISPR model identifier
# =============================================================================

crispr_gene_effect = crispr_gene_effect.rename(
    columns={"Unnamed: 0": "ModelID"}
)

In [6]:
# =============================================================================
# Characterize model overlap
# =============================================================================

consensus_model_ids = set(consensus_cellline_scores["ModelID"])
crispr_model_ids = set(crispr_gene_effect["ModelID"])

shared_model_ids = consensus_model_ids & crispr_model_ids

print("Frozen consensus models:", len(consensus_model_ids))
print("CRISPR models:", len(crispr_model_ids))
print("Shared models:", len(shared_model_ids))
print(
    "Frozen consensus models without CRISPR:",
    len(consensus_model_ids - crispr_model_ids),
)

Frozen consensus models: 713
CRISPR models: 1178
Shared models: 539
Frozen consensus models without CRISPR: 174


In [7]:
# =============================================================================
# Align inputs to the shared model cohort
# =============================================================================

shared_consensus_scores = (
    consensus_cellline_scores[
        consensus_cellline_scores["ModelID"].isin(crispr_model_ids)
    ]
    .set_index("ModelID")
    .copy()
)

shared_crispr_gene_effect = (
    crispr_gene_effect
    .set_index("ModelID")
    .loc[shared_consensus_scores.index]
    .copy()
)

In [8]:
# =============================================================================
# Parse CRISPR gene identifiers
# =============================================================================

crispr_gene_annotations = pd.DataFrame(
    {"source_gene_label": shared_crispr_gene_effect.columns}
)

parsed_gene_ids = crispr_gene_annotations["source_gene_label"].str.extract(
    r"^(?P<gene_symbol>.+) \((?P<entrez_id>\d+)\)$"
)

crispr_gene_annotations = pd.concat(
    [crispr_gene_annotations, parsed_gene_ids],
    axis=1,
)

crispr_gene_annotations["entrez_id"] = (
    pd.to_numeric(
        crispr_gene_annotations["entrez_id"],
        errors="coerce",
    )
    .astype("Int64")
)

In [9]:
# =============================================================================
# Summarize CRISPR gene identifier parsing
# =============================================================================

print("CRISPR gene labels:", len(crispr_gene_annotations))
print(
    "Unparsed labels:",
    crispr_gene_annotations["gene_symbol"].isna().sum(),
)
print(
    "Duplicate source labels:",
    crispr_gene_annotations["source_gene_label"].duplicated().sum(),
)
print(
    "Gene symbols with multiple source labels:",
    crispr_gene_annotations["gene_symbol"].duplicated(keep=False).sum(),
)
print(
    "Entrez IDs with multiple source labels:",
    crispr_gene_annotations["entrez_id"].duplicated(keep=False).sum(),
)

CRISPR gene labels: 17916
Unparsed labels: 0
Duplicate source labels: 0
Gene symbols with multiple source labels: 0
Entrez IDs with multiple source labels: 0


In [10]:
# =============================================================================
# Characterize CRISPR gene coverage
# =============================================================================

crispr_gene_coverage = crispr_gene_annotations.copy()

crispr_gene_coverage["n_observed"] = (
    shared_crispr_gene_effect.notna().sum(axis=0).to_numpy()
)

crispr_gene_coverage["n_missing"] = (
    shared_crispr_gene_effect.isna().sum(axis=0).to_numpy()
)

crispr_gene_coverage["missing_fraction"] = (
    crispr_gene_coverage["n_missing"]
    / len(shared_crispr_gene_effect)
)

In [11]:
# =============================================================================
# Summarize CRISPR gene coverage
# =============================================================================

print(
    crispr_gene_coverage[
        ["n_observed", "n_missing", "missing_fraction"]
    ].describe()
)

print(
    "\nGenes complete across shared models:",
    (crispr_gene_coverage["n_missing"] == 0).sum(),
)

         n_observed     n_missing  missing_fraction
count  17916.000000  17916.000000      17916.000000
mean     535.128544      3.871456          0.007183
std       25.369245     25.369245          0.047067
min      262.000000      0.000000          0.000000
25%      539.000000      0.000000          0.000000
50%      539.000000      0.000000          0.000000
75%      539.000000      0.000000          0.000000
max      539.000000    277.000000          0.513915

Genes complete across shared models: 17097


In [12]:
# =============================================================================
# Characterize CRISPR gene informativeness
# =============================================================================

crispr_gene_coverage["gene_effect_sd"] = (
    shared_crispr_gene_effect.std(axis=0).to_numpy()
)

crispr_gene_coverage["n_unique"] = (
    shared_crispr_gene_effect.nunique(axis=0).to_numpy()
)

In [13]:
# =============================================================================
# Summarize CRISPR gene informativeness
# =============================================================================

print(
    crispr_gene_coverage[
        ["gene_effect_sd", "n_unique"]
    ].describe()
)

print(
    "\nGenes with zero variance:",
    (crispr_gene_coverage["gene_effect_sd"] == 0).sum(),
)
print(
    "Genes with fewer than 2 observed values:",
    (crispr_gene_coverage["n_observed"] < 2).sum(),
)

       gene_effect_sd      n_unique
count    17916.000000  17916.000000
mean         0.141296    535.128544
std          0.071623     25.369245
min          0.061053    262.000000
25%          0.102247    539.000000
50%          0.115731    539.000000
75%          0.142840    539.000000
max          0.853483    539.000000

Genes with zero variance: 0
Genes with fewer than 2 observed values: 0


In [14]:
# =============================================================================
# Freeze CRISPR gene eligibility
# =============================================================================

MIN_CRISPR_MODEL_FRACTION = 0.90
MIN_CRISPR_MODELS = int(
    np.ceil(MIN_CRISPR_MODEL_FRACTION * len(shared_crispr_gene_effect))
)

crispr_gene_coverage["analysis_eligible"] = (
    (crispr_gene_coverage["n_observed"] >= MIN_CRISPR_MODELS)
    & (crispr_gene_coverage["n_unique"] >= 2)
    & (crispr_gene_coverage["gene_effect_sd"] > 0)
)

In [15]:
# =============================================================================
# Summarize frozen CRISPR gene universe
# =============================================================================

eligible_gene_count = crispr_gene_coverage["analysis_eligible"].sum()

print("Minimum CRISPR models required:", MIN_CRISPR_MODELS)
print("Eligible CRISPR genes:", eligible_gene_count)
print(
    "Excluded CRISPR genes:",
    len(crispr_gene_coverage) - eligible_gene_count,
)

Minimum CRISPR models required: 486
Eligible CRISPR genes: 17205
Excluded CRISPR genes: 711


## CRISPR harmonization and frozen analytical gene universe

The frozen Phase 4 cell-line cohort contains 713 models, of which 539 have
DepMap Public 24Q4 CRISPR gene-effect measurements. These 539 shared models
define the downstream model universe for notebook 500; CRISPR models outside
the frozen consensus cohort are not introduced.

The CRISPR matrix contains 17,916 gene columns. Original DepMap
`SYMBOL (EntrezID)` labels are retained as source analytical identifiers, with
gene symbols and Entrez IDs parsed transparently. All labels were successfully
parsed, with no duplicate source labels, gene-symbol ambiguities, or Entrez-ID
ambiguities requiring external remapping or alias rescue.

Gene eligibility was frozen before inspecting any program–dependency
association. A gene is eligible when CRISPR gene effect is observed in at least
90% of the shared model cohort (`n >= 486`) and retains non-zero variation.
No additional effect-variance threshold is imposed because all source genes
retain measurable variation in the shared cohort.

This prespecified rule retains 17,205 genes and excludes 711 genes because of
insufficient model coverage. The resulting gene universe is fixed for the
primary association family and will not be modified on the basis of downstream
effect sizes, p-values, or biological annotations.

In [16]:
# =============================================================================
# Construct lineage-aware shared cohort
# =============================================================================

shared_consensus_analysis = (
    shared_consensus_scores
    .reset_index()
    .merge(
        cell_line_cohort[
            [
                "ModelID",
                "OncotreeLineage",
            ]
        ],
        on="ModelID",
        how="left",
        validate="one_to_one",
    )
    .set_index("ModelID")
)

In [17]:
# =============================================================================
# Characterize lineage support in the shared cohort
# =============================================================================

MIN_LINEAGE_N = 15    # Freeze lineage-support parameter

shared_lineage_counts = (
    shared_consensus_analysis["OncotreeLineage"]
    .value_counts()
    .rename_axis("OncotreeLineage")
    .reset_index(name="n_models")
)

shared_lineage_counts["lineage_supported"] = (
    shared_lineage_counts["n_models"] >= MIN_LINEAGE_N
)

print("Shared lineages:", len(shared_lineage_counts))
print(
    "Lineages with n >= MIN_LINEAGE_N:",
    shared_lineage_counts["lineage_supported"].sum(),
)
print(
    "Models in supported lineages:",
    shared_lineage_counts.loc[
        shared_lineage_counts["lineage_supported"],
        "n_models",
    ].sum(),
)

Shared lineages: 26
Lineages with n >= MIN_LINEAGE_N: 13
Models in supported lineages: 439


## Lineage support and evidentiary roles

The 539-model CRISPR–consensus overlap spans 26 observed cell-line lineages.
Using the Phase 4 lineage-support threshold retained from notebook 402
(`MIN_LINEAGE_N = 15`), 13 lineages have sufficient sample support for
lineage-specific analyses, comprising 439 models.

This threshold defines eligibility for within-lineage evidence only. It is not
used to exclude smaller lineages from the primary global lineage-adjusted
association model.

Primary inference therefore uses the full shared model cohort with lineage
included as a fixed-effect adjustment variable, subject to gene-level CRISPR
missingness. Lineage-specific estimates are treated as secondary evidence for
effect consistency, heterogeneity, and lineage restriction when sample support
is sufficient.

Because the frozen consensus representations contain substantial lineage
structure, naïve pooled pan-cancer correlations are not considered primary
evidence.

In [18]:
# =============================================================================
# Freeze primary analysis universes
# =============================================================================

consensus_program_ids = shared_consensus_scores.columns.tolist()

eligible_crispr_genes = (
    crispr_gene_coverage.loc[
        crispr_gene_coverage["analysis_eligible"],
        "source_gene_label",
    ]
    .tolist()
)

supported_lineages = (
    shared_lineage_counts.loc[
        shared_lineage_counts["lineage_supported"],
        "OncotreeLineage",
    ]
    .tolist()
)

In [19]:
# =============================================================================
# Confirm lineage annotation coverage
# =============================================================================

missing_lineage_count = (
    shared_consensus_analysis["OncotreeLineage"]
    .isna()
    .sum()
)

print(
    "Shared models without lineage annotation:",
    missing_lineage_count,
)

Shared models without lineage annotation: 0


In [20]:
# =============================================================================
# Freeze primary CRISPR association parameters
# =============================================================================

PRIMARY_MIN_LINEAGE_N = 2

PRIMARY_FDR_ALPHA = 0.05
PRIMARY_FDR_METHOD = "fdr_bh"
PRIMARY_COV_TYPE = "HC3"

PRIMARY_HYPOTHESIS_COUNT = (
    len(consensus_program_ids)
    * len(eligible_crispr_genes)
)

# Lineages represented by a single observed model for a given gene do not
# contribute within-lineage information to the primary program coefficient.
# This estimability rule is distinct from MIN_LINEAGE_N = 15, which applies
# only to secondary within-lineage characterization.

# Primary model:
# gene effect ~ consensus score + lineage fixed effects
#
# Multiplicity is controlled globally across the full frozen
# program × eligible-gene hypothesis family.
#
# Pooled Spearman associations are descriptive only.

## Prespecified primary association model and multiplicity family

The primary CRISPR analysis evaluates the frozen Cartesian hypothesis family
defined by three Phase 4 consensus programs and 17,205 analysis-eligible CRISPR
genes, for a total of 51,615 program–gene tests.

For each gene, the primary model estimates the association between CRISPR gene
effect and the corresponding frozen consensus-program score while adjusting for
cell-line lineage as a fixed effect:

`gene effect ~ consensus score + lineage`

The consensus-program score is the only program-level predictor of interest.
Lineage adjustment is mandatory because substantial lineage structure was
already documented upstream and remains present in the shared CRISPR cohort.

For a given program–gene fit, only lineages represented by at least two models
with observed gene effect and program score are retained in the primary
regression. A singleton lineage provides no within-lineage information for the
program coefficient and can produce leverage equal to one under fixed-effects
encoding, making HC3 variance estimation undefined. This is an estimability
rule applied uniformly before each fit, not a biological eligibility criterion
or a result-dependent filter.

This primary estimability requirement is distinct from `MIN_LINEAGE_N = 15`,
which is used only for secondary within-lineage characterization. It does not
alter the frozen 51,615-test hypothesis family: all eligible program–gene
hypotheses remain part of the global multiplicity correction.

Heteroskedasticity-consistent HC3 standard errors are used for primary
inference. Multiple testing is controlled globally across the complete frozen
51,615-test family using the Benjamini–Hochberg false-discovery-rate procedure
at `alpha = 0.05`.

Pooled pan-cancer correlations, if reported, are descriptive only and do not
constitute primary inferential evidence. Within-lineage estimates and
cross-lineage consistency analyses are secondary characterization layers and
will not be used to redefine the primary hypothesis family.

No additional proliferation covariate is introduced because no frozen,
provenance-supported cell-line proliferation representation is available for
this analysis. This unresolved source of confounding is retained as a
limitation rather than addressed through a post-hoc proxy.

In [22]:
# =============================================================================
# Define primary lineage-adjusted association model
# =============================================================================

def fit_lineage_adjusted_association(
    gene_effect,
    program_score,
):
    observed = gene_effect.notna() & program_score.notna()

    observed_lineages = shared_consensus_analysis.loc[
        observed,
        "OncotreeLineage",
    ]

    lineage_counts = observed_lineages.value_counts()

    retained_lineages = lineage_counts.loc[
        lineage_counts >= PRIMARY_MIN_LINEAGE_N
    ].index

    retained = (
        observed
        & shared_consensus_analysis["OncotreeLineage"].isin(
            retained_lineages
        )
    )

    lineage_design_local = pd.get_dummies(
        shared_consensus_analysis.loc[
            retained,
            "OncotreeLineage",
        ],
        prefix="lineage",
        drop_first=True,
        dtype=float,
    )

    design = pd.concat(
        [
            program_score.loc[retained].rename("program_score"),
            lineage_design_local,
        ],
        axis=1,
    )

    design = sm.add_constant(
        design,
        has_constant="add",
    )

    model = sm.OLS(
        gene_effect.loc[retained],
        design,
    ).fit(cov_type=PRIMARY_COV_TYPE)

    return {
        "n_models": int(retained.sum()),
        "beta": model.params["program_score"],
        "standard_error": model.bse["program_score"],
        "p_value": model.pvalues["program_score"],
    }

In [23]:
# =============================================================================
# Construct primary CRISPR analysis matrix
# =============================================================================

primary_crispr_gene_effect = (
    shared_crispr_gene_effect[
        eligible_crispr_genes
    ]
    .copy()
)

In [24]:
# =============================================================================
# Run primary lineage-adjusted CRISPR association screen
# =============================================================================

primary_association_records = []

for program_id in consensus_program_ids:
    program_score = shared_consensus_analysis[program_id]

    for gene_label in eligible_crispr_genes:
        gene_effect = primary_crispr_gene_effect[gene_label]

        association = fit_lineage_adjusted_association(
            gene_effect=gene_effect,
            program_score=program_score,
        )

        primary_association_records.append(
            {
                "consensus_program_id": program_id,
                "source_gene_label": gene_label,
                **association,
            }
        )

primary_crispr_associations = pd.DataFrame(
    primary_association_records
)

In [25]:
# =============================================================================
# Verify primary CRISPR association numerical stability
# =============================================================================

print(
    "Primary associations:",
    len(primary_crispr_associations),
)
print(
    "Non-finite betas:",
    (~np.isfinite(primary_crispr_associations["beta"])).sum(),
)
print(
    "Non-finite standard errors:",
    (~np.isfinite(
        primary_crispr_associations["standard_error"]
    )).sum(),
)
print(
    "Non-finite p-values:",
    (~np.isfinite(primary_crispr_associations["p_value"])).sum(),
)

Primary associations: 51615
Non-finite betas: 0
Non-finite standard errors: 0
Non-finite p-values: 0


In [26]:
# =============================================================================
# Apply global primary multiple-testing correction
# =============================================================================

primary_fdr_reject, primary_fdr_q_values, _, _ = multipletests(
    primary_crispr_associations["p_value"],
    alpha=PRIMARY_FDR_ALPHA,
    method=PRIMARY_FDR_METHOD,
)

primary_crispr_associations["fdr_q_value"] = primary_fdr_q_values
primary_crispr_associations["fdr_significant"] = primary_fdr_reject

In [27]:
# =============================================================================
# Summarize primary multiple-testing results
# =============================================================================

primary_fdr_summary = (
    primary_crispr_associations
    .groupby("consensus_program_id")
    .agg(
        n_tests=("source_gene_label", "size"),
        n_fdr_significant=("fdr_significant", "sum"),
        min_fdr_q_value=("fdr_q_value", "min"),
    )
    .reset_index()
)

print(
    "Primary FDR-significant associations:",
    primary_crispr_associations["fdr_significant"].sum(),
    "of",
    len(primary_crispr_associations),
)

primary_fdr_summary

Primary FDR-significant associations: 944 of 51615


,consensus_program_id,n_tests,n_fdr_significant,min_fdr_q_value
0,CONSENSUS_TX_01,17205,324,3.920108e-08
1,CONSENSUS_TX_02,17205,256,6.611589e-17
2,CONSENSUS_TX_03,17205,364,1.962135e-08


## Primary lineage-adjusted association screen

The prespecified primary screen evaluated 51,615 lineage-adjusted
program–gene associations across the three frozen consensus programs and
17,205 eligible CRISPR genes.

After global Benjamini–Hochberg correction across the complete hypothesis
family, 944 associations met `FDR < 0.05`:

- `CONSENSUS_TX_01`: 324 associations
- `CONSENSUS_TX_02`: 256 associations
- `CONSENSUS_TX_03`: 364 associations

These results indicate that the frozen consensus representations are associated
with systematic variation in CRISPR gene effect after adjustment for cell-line
lineage.

FDR significance alone does not define a putative vulnerability. Direction and
effect magnitude must be considered explicitly: a negative consensus-score
coefficient indicates that higher program activity is associated with more
negative CRISPR gene effect, whereas a positive coefficient indicates the
opposite pattern.

The observed associations remain computational dependency associations within
the overlapping DepMap model cohort. They do not establish causal dependency,
target validation, therapeutic efficacy, or independent cross-dataset
replication.

In [28]:
# =============================================================================
# Prepare FDR-significant primary associations
# =============================================================================

significant_primary_associations = (
    primary_crispr_associations.loc[
        primary_crispr_associations["fdr_significant"]
    ]
    .copy()
)

significant_primary_associations["effect_direction"] = np.where(
    significant_primary_associations["beta"] < 0,
    "higher_program_stronger_dependency",
    "higher_program_weaker_dependency",
)

In [29]:
# =============================================================================
# Summarize significant primary effect magnitudes
# =============================================================================

significant_primary_associations["absolute_beta"] = (
    significant_primary_associations["beta"].abs()
)

primary_effect_summary = (
    significant_primary_associations
    .groupby(
        [
            "consensus_program_id",
            "effect_direction",
        ]
    )
    .agg(
        n_associations=("source_gene_label", "size"),
        median_beta=("beta", "median"),
        median_absolute_beta=("absolute_beta", "median"),
        max_absolute_beta=("absolute_beta", "max"),
    )
    .reset_index()
)

primary_effect_summary

,consensus_program_id,effect_direction,n_associations,median_beta,median_absolute_beta,max_absolute_beta
0,CONSENSUS_TX_01,higher_program_stronger_dependency,232,-0.141864,0.141864,0.335312
1,CONSENSUS_TX_01,higher_program_weaker_dependency,92,0.114236,0.114236,0.439010
2,CONSENSUS_TX_02,higher_program_stronger_dependency,114,-0.048981,0.048981,0.207240
3,CONSENSUS_TX_02,higher_program_weaker_dependency,142,0.057534,0.057534,0.233205
4,CONSENSUS_TX_03,higher_program_stronger_dependency,146,-0.060816,0.060816,0.250815
5,CONSENSUS_TX_03,higher_program_weaker_dependency,218,0.055445,0.055445,0.194716


## Direction and magnitude of primary CRISPR associations

The 944 FDR-significant primary associations occur in both effect directions.

Associations in which higher consensus-program score is associated with
stronger dependency (`beta < 0`) comprise:

- 232 associations for `CONSENSUS_TX_01`
- 114 associations for `CONSENSUS_TX_02`
- 146 associations for `CONSENSUS_TX_03`

Associations in the opposite direction (`beta > 0`) are also frequent,
indicating that statistical significance cannot be interpreted generically as
functional vulnerability evidence.

Effect magnitudes differ across programs. Among significant stronger-dependency
associations, the median lineage-adjusted coefficient is approximately
`-0.142` for `CONSENSUS_TX_01`, compared with `-0.049` and `-0.061` for
`CONSENSUS_TX_02` and `CONSENSUS_TX_03`, respectively.

These effect-size distributions are descriptive and will not be used to define
a post-hoc magnitude threshold. Gene-level interpretation requires additional
assessment of lineage-specific support and cross-lineage consistency before
individual associations are treated as candidate putative vulnerability
signals.

In [30]:
# =============================================================================
# Define within-lineage association estimator
# =============================================================================

def estimate_within_lineage_association(
    gene_effect,
    program_score,
    lineage,
):
    lineage_mask = (
        shared_consensus_analysis["OncotreeLineage"]
        == lineage
    )

    observed = (
        lineage_mask
        & gene_effect.notna()
        & program_score.notna()
    )

    x = program_score.loc[observed]
    y = gene_effect.loc[observed]

    if len(x) < MIN_LINEAGE_N:
        return None

    x_centered = x - x.mean()
    denominator = np.sum(x_centered**2)

    if denominator == 0:
        return None

    beta = np.sum(
        x_centered * (y - y.mean())
    ) / denominator

    return {
        "OncotreeLineage": lineage,
        "n_models": len(x),
        "beta": beta,
    }

In [31]:
# =============================================================================
# Estimate within-lineage effects for primary significant associations
# =============================================================================

within_lineage_records = []

for row in significant_primary_associations.itertuples(index=False):
    program_score = shared_consensus_analysis[
        row.consensus_program_id
    ]
    gene_effect = primary_crispr_gene_effect[
        row.source_gene_label
    ]

    for lineage in supported_lineages:
        estimate = estimate_within_lineage_association(
            gene_effect=gene_effect,
            program_score=program_score,
            lineage=lineage,
        )

        if estimate is not None:
            within_lineage_records.append(
                {
                    "consensus_program_id": row.consensus_program_id,
                    "source_gene_label": row.source_gene_label,
                    **estimate,
                }
            )

within_lineage_associations = pd.DataFrame(
    within_lineage_records
)

In [32]:
# =============================================================================
# Summarize cross-lineage effect consistency
# =============================================================================

primary_effect_lookup = (
    significant_primary_associations[
        [
            "consensus_program_id",
            "source_gene_label",
            "beta",
        ]
    ]
    .rename(columns={"beta": "primary_beta"})
)

within_lineage_summary = (
    within_lineage_associations
    .merge(
        primary_effect_lookup,
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="many_to_one",
    )
)

within_lineage_summary["direction_consistent"] = (
    np.sign(within_lineage_summary["beta"])
    == np.sign(within_lineage_summary["primary_beta"])
)

cross_lineage_consistency = (
    within_lineage_summary
    .groupby(
        [
            "consensus_program_id",
            "source_gene_label",
        ]
    )
    .agg(
        n_evaluable_lineages=("OncotreeLineage", "nunique"),
        n_direction_consistent=("direction_consistent", "sum"),
        median_lineage_beta=("beta", "median"),
        min_lineage_beta=("beta", "min"),
        max_lineage_beta=("beta", "max"),
    )
    .reset_index()
)

cross_lineage_consistency["direction_consistency_fraction"] = (
    cross_lineage_consistency["n_direction_consistent"]
    / cross_lineage_consistency["n_evaluable_lineages"]
)

In [33]:
# =============================================================================
# Summarize cross-lineage consistency distribution
# =============================================================================

cross_lineage_consistency = (
    cross_lineage_consistency
    .merge(
        significant_primary_associations[
            [
                "consensus_program_id",
                "source_gene_label",
                "effect_direction",
            ]
        ],
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

cross_lineage_consistency_summary = (
    cross_lineage_consistency
    .groupby(
        [
            "consensus_program_id",
            "effect_direction",
        ]
    )
    .agg(
        n_associations=("source_gene_label", "size"),
        median_evaluable_lineages=("n_evaluable_lineages", "median"),
        min_evaluable_lineages=("n_evaluable_lineages", "min"),
        median_direction_consistency=(
            "direction_consistency_fraction",
            "median",
        ),
        fully_direction_consistent=(
            "direction_consistency_fraction",
            lambda x: (x == 1).sum(),
        ),
    )
    .reset_index()
)

cross_lineage_consistency_summary

,consensus_program_id,effect_direction,n_associations,median_evaluable_lineages,min_evaluable_lineages,median_direction_consistency,fully_direction_consistent
0,CONSENSUS_TX_01,higher_program_stronger_dependency,232,13.0,13,0.846154,13
1,CONSENSUS_TX_01,higher_program_weaker_dependency,92,13.0,13,0.807692,4
2,CONSENSUS_TX_02,higher_program_stronger_dependency,114,13.0,13,0.692308,1
3,CONSENSUS_TX_02,higher_program_weaker_dependency,142,13.0,13,0.769231,2
4,CONSENSUS_TX_03,higher_program_stronger_dependency,146,13.0,13,0.769231,7
5,CONSENSUS_TX_03,higher_program_weaker_dependency,218,13.0,13,0.769231,5


## Cross-lineage directional consistency

All 944 primary FDR-significant associations were evaluable in each of the
13 lineages meeting the prespecified `MIN_LINEAGE_N = 15` requirement after
gene-specific CRISPR missingness was taken into account.

Directional consistency varies across programs and association directions.
Median agreement between within-lineage effects and the corresponding primary
lineage-adjusted coefficient ranges from approximately 69% to 85% of evaluable
lineages.

`CONSENSUS_TX_01` shows the highest median directional consistency, particularly
for stronger-dependency associations (`beta < 0`), for which the median
association agrees in direction across 11 of 13 evaluable lineages
(`0.846`). The corresponding median consistency is lower for
`CONSENSUS_TX_02` and intermediate for `CONSENSUS_TX_03`.

Complete directional concordance across all 13 lineages is uncommon:
13 stronger-dependency associations for `CONSENSUS_TX_01`, one for
`CONSENSUS_TX_02`, and seven for `CONSENSUS_TX_03` show fully consistent
within-lineage direction. Complete concordance is likewise uncommon among
weaker-dependency associations.

These analyses characterize heterogeneity within the same DepMap model system;
they are not independent replication and do not constitute a secondary
significance gate. Associations with heterogeneous lineage effects remain part
of the primary result set, while cross-lineage consistency provides an
additional descriptive dimension for subsequent interpretation.

In [34]:
# =============================================================================
# Quantify cross-lineage effect heterogeneity
# =============================================================================

cross_lineage_heterogeneity = (
    within_lineage_associations
    .groupby(
        [
            "consensus_program_id",
            "source_gene_label",
        ]
    )
    .agg(
        lineage_beta_sd=("beta", "std"),
        lineage_beta_q25=("beta", lambda x: x.quantile(0.25)),
        lineage_beta_q75=("beta", lambda x: x.quantile(0.75)),
        lineage_beta_min=("beta", "min"),
        lineage_beta_max=("beta", "max"),
    )
    .reset_index()
)

cross_lineage_heterogeneity["lineage_beta_iqr"] = (
    cross_lineage_heterogeneity["lineage_beta_q75"]
    - cross_lineage_heterogeneity["lineage_beta_q25"]
)

cross_lineage_heterogeneity["lineage_beta_range"] = (
    cross_lineage_heterogeneity["lineage_beta_max"]
    - cross_lineage_heterogeneity["lineage_beta_min"]
)

In [35]:
# =============================================================================
# Summarize cross-lineage effect heterogeneity
# =============================================================================

cross_lineage_heterogeneity = (
    cross_lineage_heterogeneity
    .merge(
        significant_primary_associations[
            [
                "consensus_program_id",
                "source_gene_label",
                "effect_direction",
            ]
        ],
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

cross_lineage_heterogeneity_summary = (
    cross_lineage_heterogeneity
    .groupby(
        [
            "consensus_program_id",
            "effect_direction",
        ]
    )
    .agg(
        n_associations=("source_gene_label", "size"),
        median_lineage_beta_sd=("lineage_beta_sd", "median"),
        median_lineage_beta_iqr=("lineage_beta_iqr", "median"),
        median_lineage_beta_range=("lineage_beta_range", "median"),
    )
    .reset_index()
)

cross_lineage_heterogeneity_summary

,consensus_program_id,effect_direction,n_associations,median_lineage_beta_sd,median_lineage_beta_iqr,median_lineage_beta_range
0,CONSENSUS_TX_01,higher_program_stronger_dependency,232,0.219246,0.222571,0.762071
1,CONSENSUS_TX_01,higher_program_weaker_dependency,92,0.198588,0.182048,0.706759
2,CONSENSUS_TX_02,higher_program_stronger_dependency,114,0.137957,0.086975,0.521416
3,CONSENSUS_TX_02,higher_program_weaker_dependency,142,0.144658,0.100825,0.516871
4,CONSENSUS_TX_03,higher_program_stronger_dependency,146,0.091324,0.084907,0.342695
5,CONSENSUS_TX_03,higher_program_weaker_dependency,218,0.097935,0.082816,0.360211


## Cross-lineage effect-magnitude heterogeneity

Within-lineage effect magnitudes show substantial heterogeneity even among
associations that passed the primary global FDR threshold.

`CONSENSUS_TX_01` has the largest typical between-lineage dispersion, with
median lineage-effect standard deviations of approximately `0.20–0.22` and
median effect ranges of approximately `0.71–0.76`. This is consistent with the
larger absolute primary coefficients observed for this program, but also
indicates that its stronger overall associations frequently vary considerably
in magnitude across lineage contexts.

`CONSENSUS_TX_02` shows intermediate heterogeneity, whereas
`CONSENSUS_TX_03` has the smallest median between-lineage dispersion among the
three programs.

These quantities describe effect-magnitude variability and are not formal
heterogeneity tests or independent validation statistics. No post-hoc
heterogeneity cutoff is introduced. Directional consistency and effect
dispersion will therefore be retained as continuous descriptive evidence
dimensions rather than used to rescue or exclude primary associations.

The combination of primary lineage-adjusted significance, effect direction,
within-lineage directional consistency, and between-lineage magnitude
heterogeneity will be carried forward for gene-level interpretation.

In [36]:
# =============================================================================
# Integrate primary and cross-lineage association evidence
# =============================================================================

primary_significant_evidence = (
    significant_primary_associations
    .merge(
        cross_lineage_consistency[
            [
                "consensus_program_id",
                "source_gene_label",
                "n_evaluable_lineages",
                "n_direction_consistent",
                "direction_consistency_fraction",
                "median_lineage_beta",
            ]
        ],
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        cross_lineage_heterogeneity[
            [
                "consensus_program_id",
                "source_gene_label",
                "lineage_beta_sd",
                "lineage_beta_iqr",
                "lineage_beta_range",
            ]
        ],
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [37]:
# =============================================================================
# Add CRISPR gene annotations to integrated evidence
# =============================================================================

primary_significant_evidence = (
    primary_significant_evidence
    .merge(
        crispr_gene_annotations[
            [
                "source_gene_label",
                "gene_symbol",
                "entrez_id",
            ]
        ],
        on="source_gene_label",
        how="left",
        validate="many_to_one",
    )
)

In [38]:
# =============================================================================
# Characterize gene-level overlap across consensus programs
# =============================================================================

significant_gene_program_count = (
    primary_significant_evidence
    .groupby("source_gene_label")["consensus_program_id"]
    .nunique()
)

print(
    "Unique genes with primary FDR-significant associations:",
    significant_gene_program_count.size,
)

print(
    "Genes associated with one program:",
    (significant_gene_program_count == 1).sum(),
)

print(
    "Genes associated with two programs:",
    (significant_gene_program_count == 2).sum(),
)

print(
    "Genes associated with all three programs:",
    (significant_gene_program_count == 3).sum(),
)

Unique genes with primary FDR-significant associations: 819
Genes associated with one program: 696
Genes associated with two programs: 121
Genes associated with all three programs: 2


## Gene-level overlap across consensus programs

The 944 primary FDR-significant program–gene associations correspond to 819
unique CRISPR genes.

Most associated genes are program-restricted: 696 genes are associated with
only one consensus program. A smaller subset shows evidence across multiple
program contexts, with 121 genes associated with two programs and only two
genes associated with all three frozen consensus programs.

Accordingly, the association count should not be interpreted as a count of
independent genes. Multi-program associations represent repeated evidence for
the same CRISPR dependency phenotype across distinct frozen transcriptomic
contexts.

Program overlap is retained as a descriptive property rather than used as a
priority or validation criterion. Association with multiple consensus programs
does not by itself imply a shared causal mechanism or a more strongly supported
functional target.

In [39]:
# =============================================================================
# Characterize effect direction across multi-program genes
# =============================================================================

multi_program_gene_direction = (
    primary_significant_evidence
    .groupby("source_gene_label")
    .agg(
        n_programs=("consensus_program_id", "nunique"),
        n_effect_directions=("effect_direction", "nunique"),
    )
    .reset_index()
)

multi_program_gene_direction = (
    multi_program_gene_direction.loc[
        multi_program_gene_direction["n_programs"] > 1
    ]
    .copy()
)

multi_program_gene_direction["direction_consistent_across_programs"] = (
    multi_program_gene_direction["n_effect_directions"] == 1
)

print("Multi-program genes:", len(multi_program_gene_direction))
print(
    "Same direction across programs:",
    multi_program_gene_direction[
        "direction_consistent_across_programs"
    ].sum(),
)
print(
    "Opposite directions across programs:",
    (
        ~multi_program_gene_direction[
            "direction_consistent_across_programs"
        ]
    ).sum(),
)

Multi-program genes: 123
Same direction across programs: 60
Opposite directions across programs: 63


## Directional behavior of multi-program CRISPR associations

Among the 123 genes with primary FDR-significant associations to more than one
consensus program, 60 retain the same effect direction across programs whereas
63 show opposite directions in different program contexts.

Thus, multi-program association does not imply a uniform dependency phenotype.
For approximately half of the shared genes, higher activity of one frozen
consensus program is associated with stronger CRISPR dependency while higher
activity of another is associated with weaker dependency.

This pattern supports program-specific interpretation of CRISPR associations.
Gene overlap across consensus representations is therefore treated as a
descriptive property of the dependency landscape rather than as evidence for a
shared functional mechanism or a stronger vulnerability claim.

Subsequent interpretation will preserve the program–gene association as the
primary analytical unit rather than collapsing evidence to the gene level.

In [40]:
# =============================================================================
# Calculate descriptive pooled associations
# =============================================================================

pooled_association_records = []

for row in primary_significant_evidence.itertuples(index=False):
    gene_effect = primary_crispr_gene_effect[
        row.source_gene_label
    ]
    program_score = shared_consensus_analysis[
        row.consensus_program_id
    ]

    observed = gene_effect.notna() & program_score.notna()

    pooled_spearman_rho, _ = spearmanr(
        program_score.loc[observed],
        gene_effect.loc[observed],
    )

    pooled_association_records.append(
        {
            "consensus_program_id": row.consensus_program_id,
            "source_gene_label": row.source_gene_label,
            "pooled_spearman_rho": pooled_spearman_rho,
        }
    )

pooled_primary_associations = pd.DataFrame(
    pooled_association_records
)

In [41]:
# =============================================================================
# Add descriptive pooled associations to integrated evidence
# =============================================================================

primary_significant_evidence = (
    primary_significant_evidence
    .merge(
        pooled_primary_associations,
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [42]:
# =============================================================================
# Compare pooled and lineage-adjusted effect directions
# =============================================================================

primary_significant_evidence["pooled_adjusted_direction_consistent"] = (
    np.sign(primary_significant_evidence["pooled_spearman_rho"])
    == np.sign(primary_significant_evidence["beta"])
)

pooled_adjusted_direction_summary = (
    primary_significant_evidence
    .groupby("consensus_program_id")
    .agg(
        n_associations=("source_gene_label", "size"),
        n_direction_consistent=(
            "pooled_adjusted_direction_consistent",
            "sum",
        ),
        direction_consistency_fraction=(
            "pooled_adjusted_direction_consistent",
            "mean",
        ),
    )
    .reset_index()
)

pooled_adjusted_direction_summary

,consensus_program_id,n_associations,n_direction_consistent,direction_consistency_fraction
0,CONSENSUS_TX_01,324,320,0.987654
1,CONSENSUS_TX_02,256,256,1.000000
2,CONSENSUS_TX_03,364,363,0.997253


## Pooled versus lineage-adjusted association direction

Among the 944 associations identified by the prespecified lineage-adjusted
primary analysis, pooled Spearman correlations and lineage-adjusted coefficients
show highly concordant effect directions.

Directional agreement is observed for:

- 320 of 324 associations for `CONSENSUS_TX_01`
- all 256 associations for `CONSENSUS_TX_02`
- 363 of 364 associations for `CONSENSUS_TX_03`

Overall, 939 of 944 primary associations retain the same direction in the
descriptive pooled analysis.

This high directional concordance does not make pooled pan-cancer correlation
an interchangeable primary analysis. The comparison is conditional on
associations already identified using the lineage-adjusted model, and pooled
Spearman coefficients and lineage-adjusted regression coefficients are not
directly comparable effect-size measures.

Lineage therefore remains a required adjustment variable for primary inference.
The pooled correlations are retained only as descriptive context for assessing
whether the gross direction of an association is sensitive to lineage
adjustment.

In [43]:
# =============================================================================
# Identify lineage-sensitive direction reversals
# =============================================================================

lineage_sensitive_direction_reversals = (
    primary_significant_evidence.loc[
        ~primary_significant_evidence[
            "pooled_adjusted_direction_consistent"
        ],
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "beta",
            "fdr_q_value",
            "pooled_spearman_rho",
            "direction_consistency_fraction",
            "lineage_beta_sd",
        ],
    ]
    .copy()
)

lineage_sensitive_direction_reversals

,consensus_program_id,source_gene_label,gene_symbol,beta,fdr_q_value,pooled_spearman_rho,direction_consistency_fraction,lineage_beta_sd
11,CONSENSUS_TX_01,AMBN (258),AMBN,-0.056238,0.030078,0.017888,0.692308,0.076431
30,CONSENSUS_TX_01,CEPT1 (10390),CEPT1,-0.182474,0.005544,0.013567,0.615385,0.408984
142,CONSENSUS_TX_01,MEF2B (100271849),MEF2B,-0.208607,0.000413,0.019618,0.461538,0.159461
253,CONSENSUS_TX_01,RIPPLY2 (134701),RIPPLY2,0.042972,0.036085,-0.032966,0.615385,0.078380
818,CONSENSUS_TX_03,PDCD6IP (10015),PDCD6IP,0.063079,0.037868,-0.097311,0.615385,0.116919


## Lineage-sensitive direction reversals

Five of the 944 primary FDR-significant associations reverse direction between
the descriptive pooled Spearman correlation and the lineage-adjusted primary
coefficient.

The affected associations are:

- `AMBN` with `CONSENSUS_TX_01`
- `CEPT1` with `CONSENSUS_TX_01`
- `MEF2B` with `CONSENSUS_TX_01`
- `RIPPLY2` with `CONSENSUS_TX_01`
- `PDCD6IP` with `CONSENSUS_TX_03`

These reversals represent a small fraction of the primary result set, but they
illustrate cases in which pooled association direction is sensitive to lineage
composition.

Several of these associations also show limited within-lineage directional
consistency or substantial between-lineage effect dispersion. They should
therefore be interpreted as lineage-sensitive computational associations rather
than generalized pan-cancer dependency relationships.

The reversals do not invalidate the corresponding primary associations, because
primary inference was prespecified using lineage-adjusted models. Instead, they
are retained as an explicit caution against interpreting pooled correlations as
primary evidence.

In [44]:
# =============================================================================
# Construct lineage-specific profiles for direction reversals
# =============================================================================

lineage_sensitive_profiles = (
    within_lineage_associations
    .merge(
        lineage_sensitive_direction_reversals[
            [
                "consensus_program_id",
                "source_gene_label",
                "gene_symbol",
                "beta",
                "pooled_spearman_rho",
            ]
        ].rename(columns={"beta": "primary_beta"}),
        on=[
            "consensus_program_id",
            "source_gene_label",
        ],
        how="inner",
        validate="many_to_one",
    )
)

lineage_sensitive_profiles["direction_consistent_with_primary"] = (
    np.sign(lineage_sensitive_profiles["beta"])
    == np.sign(lineage_sensitive_profiles["primary_beta"])
)

In [45]:
# =============================================================================
# Summarize lineage contributions to direction reversals
# =============================================================================

lineage_sensitive_profiles["primary_opposing_lineage"] = (
    lineage_sensitive_profiles["OncotreeLineage"].where(
        ~lineage_sensitive_profiles[
            "direction_consistent_with_primary"
        ]
    )
)

lineage_sensitive_summary = (
    lineage_sensitive_profiles
    .groupby(
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
        ]
    )
    .agg(
        primary_beta=("primary_beta", "first"),
        pooled_spearman_rho=("pooled_spearman_rho", "first"),
        n_evaluable_lineages=("OncotreeLineage", "nunique"),
        n_direction_consistent=(
            "direction_consistent_with_primary",
            "sum",
        ),
        n_direction_opposed=(
            "direction_consistent_with_primary",
            lambda x: (~x).sum(),
        ),
        opposing_lineages=(
            "primary_opposing_lineage",
            lambda x: "; ".join(x.dropna()),
        ),
    )
    .reset_index()
)

lineage_sensitive_summary

,consensus_program_id,source_gene_label,gene_symbol,primary_beta,pooled_spearman_rho,n_evaluable_lineages,n_direction_consistent,n_direction_opposed,opposing_lineages
0,CONSENSUS_TX_01,AMBN (258),AMBN,-0.056238,0.017888,13,9,4,CNS/Brain; Pancreas; Peripheral Nervous System...
1,CONSENSUS_TX_01,CEPT1 (10390),CEPT1,-0.182474,0.013567,13,8,5,Esophagus/Stomach; Bowel; Breast; Pancreas; Skin
2,CONSENSUS_TX_01,MEF2B (100271849),MEF2B,-0.208607,0.019618,13,6,7,Lung; Esophagus/Stomach; CNS/Brain; Breast; He...
3,CONSENSUS_TX_01,RIPPLY2 (134701),RIPPLY2,0.042972,-0.032966,13,8,5,Ovary/Fallopian Tube; Pancreas; Skin; Peripher...
4,CONSENSUS_TX_03,PDCD6IP (10015),PDCD6IP,0.063079,-0.097311,13,8,5,Lung; Lymphoid; Bowel; Pancreas; Myeloid


## Lineage composition of pooled-direction reversals

The five pooled-versus-adjusted direction reversals are not attributable to a
single isolated lineage. Each association shows an opposing within-lineage
effect direction in multiple lineage contexts.

Directional agreement with the primary lineage-adjusted coefficient ranges
from 6 to 9 of the 13 evaluable lineages. `MEF2B` with `CONSENSUS_TX_01` is the
most heterogeneous case, with only 6 of 13 lineages agreeing with the primary
effect direction.

The specific opposing lineages differ across genes, indicating that the
observed reversals reflect gene- and program-specific lineage composition
rather than one universal lineage confounder.

These profiles reinforce the interpretation of the five associations as
lineage-sensitive computational dependency associations. They remain part of
the prespecified primary result set but require explicit caution against a
generalized pan-cancer interpretation.

In [46]:
# =============================================================================
# Construct integrated primary CRISPR evidence table
# =============================================================================

primary_significant_evidence["n_significant_programs_for_gene"] = (
    primary_significant_evidence["source_gene_label"]
    .map(significant_gene_program_count)
)

primary_significant_evidence["lineage_sensitive_direction_reversal"] = (
    ~primary_significant_evidence[
        "pooled_adjusted_direction_consistent"
    ]
)

crispr_primary_evidence_table = (
    primary_significant_evidence[
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "n_models",
            "beta",
            "standard_error",
            "p_value",
            "fdr_q_value",
            "effect_direction",
            "absolute_beta",
            "n_evaluable_lineages",
            "n_direction_consistent",
            "direction_consistency_fraction",
            "median_lineage_beta",
            "lineage_beta_sd",
            "lineage_beta_iqr",
            "lineage_beta_range",
            "pooled_spearman_rho",
            "pooled_adjusted_direction_consistent",
            "lineage_sensitive_direction_reversal",
            "n_significant_programs_for_gene",
        ]
    ]
    .copy()
)

In [47]:
# =============================================================================
# Define putative vulnerability association subset
# =============================================================================

crispr_putative_vulnerability_associations = (
    crispr_primary_evidence_table.loc[
        crispr_primary_evidence_table["effect_direction"]
        == "higher_program_stronger_dependency"
    ]
    .copy()
)

In [48]:
# =============================================================================
# Characterize putative vulnerability gene overlap
# =============================================================================

putative_vulnerability_gene_program_count = (
    crispr_putative_vulnerability_associations
    .groupby("source_gene_label")["consensus_program_id"]
    .nunique()
)

print(
    "Putative vulnerability associations:",
    len(crispr_putative_vulnerability_associations),
)

print(
    "Unique putative vulnerability genes:",
    putative_vulnerability_gene_program_count.size,
)

print(
    "Genes associated with one program:",
    (putative_vulnerability_gene_program_count == 1).sum(),
)

print(
    "Genes associated with two programs:",
    (putative_vulnerability_gene_program_count == 2).sum(),
)

print(
    "Genes associated with all three programs:",
    (putative_vulnerability_gene_program_count == 3).sum(),
)

Putative vulnerability associations: 492
Unique putative vulnerability genes: 463
Genes associated with one program: 435
Genes associated with two programs: 27
Genes associated with all three programs: 1


## Putative vulnerability-oriented CRISPR associations

Among the 944 primary FDR-significant program–gene associations, 492 have the
direction compatible with putative functional vulnerability evidence: higher
consensus-program score is associated with more negative CRISPR gene effect.

These 492 associations correspond to 463 unique genes. Most genes are
program-specific in this direction:

- 435 genes are associated with one consensus program;
- 27 genes are associated with two consensus programs;
- one gene is associated with all three programs.

This subset is defined by the prespecified primary FDR result and effect
direction only. No additional threshold based on effect magnitude,
cross-lineage consistency, heterogeneity, biological annotation, or
multi-program recurrence is introduced.

Accordingly, membership in this subset represents computational evidence
compatible with a putative functional vulnerability in a specific frozen
program context. It does not establish causal dependency, target validation,
therapeutic efficacy, or a shared vulnerability across programs.

In [49]:
# =============================================================================
# Summarize putative vulnerability evidence by program
# =============================================================================

putative_vulnerability_summary = (
    crispr_putative_vulnerability_associations
    .groupby("consensus_program_id")
    .agg(
        n_associations=("source_gene_label", "size"),
        n_unique_genes=("source_gene_label", "nunique"),
        median_beta=("beta", "median"),
        median_absolute_beta=("absolute_beta", "median"),
        median_direction_consistency=(
            "direction_consistency_fraction",
            "median",
        ),
        median_lineage_beta_sd=("lineage_beta_sd", "median"),
        median_lineage_beta_iqr=("lineage_beta_iqr", "median"),
        median_lineage_beta_range=("lineage_beta_range", "median"),
    )
    .reset_index()
)

putative_vulnerability_summary

,consensus_program_id,n_associations,n_unique_genes,median_beta,median_absolute_beta,median_direction_consistency,median_lineage_beta_sd,median_lineage_beta_iqr,median_lineage_beta_range
0,CONSENSUS_TX_01,232,232,-0.141864,0.141864,0.846154,0.219246,0.222571,0.762071
1,CONSENSUS_TX_02,114,114,-0.048981,0.048981,0.692308,0.137957,0.086975,0.521416
2,CONSENSUS_TX_03,146,146,-0.060816,0.060816,0.769231,0.091324,0.084907,0.342695


## Program-level summary of putative vulnerability-oriented evidence

The 492 primary associations oriented toward stronger dependency are distributed
across all three frozen consensus programs, but their effect profiles differ.

`CONSENSUS_TX_01` contributes the largest number of associations and shows the
largest median absolute lineage-adjusted effect (`|beta| ≈ 0.142`). It also has
the highest median directional consistency across evaluable lineages
(`≈ 0.85`). However, this program simultaneously shows the greatest
between-lineage effect dispersion, indicating that stronger overall effects do
not imply homogeneous magnitude across lineage contexts.

`CONSENSUS_TX_02` shows the smallest median primary effect
(`|beta| ≈ 0.049`) and the lowest median directional consistency
(`≈ 0.69`), with intermediate between-lineage heterogeneity.

`CONSENSUS_TX_03` shows modest median primary effects
(`|beta| ≈ 0.061`) but comparatively lower between-lineage dispersion and
intermediate directional consistency.

These differences are descriptive properties of the frozen program-specific
CRISPR association landscapes. No program-ranking rule, effect-size cutoff,
consistency threshold, or heterogeneity gate is introduced on the basis of
these observed distributions.

Subsequent gene-level interpretation will therefore retain primary effect size,
global FDR support, cross-lineage directional consistency, and heterogeneity as
separate evidence dimensions rather than combining them into a post-hoc score.

---

## Analytical checkpoint: primary inference versus secondary characterization

The confirmatory component of notebook 500 is complete at the level of the
prespecified primary CRISPR association screen.

Primary inference was defined before inspecting program–dependency results and
consists of the frozen three-program universe, the analysis-eligible CRISPR gene
universe, lineage-adjusted regression, HC3 covariance estimation, and global
Benjamini–Hochberg correction across the complete 51,615-test hypothesis
family.

All analyses performed after identification of the primary FDR-significant
associations are secondary, result-conditioned characterization. These include
effect-direction summaries, within-lineage directional consistency,
between-lineage effect dispersion, pooled-versus-adjusted comparisons,
multi-program overlap, and characterization of the stronger-dependency
association subset.

These secondary analyses provide context for interpretation but do not redefine
primary significance, gene eligibility, program eligibility, or the
multiple-testing family. They are not independent validation and will not be
used to introduce post-hoc significance, effect-size, consistency,
heterogeneity, or recurrence thresholds.

## Prespecified gene-level interpretation boundary

Gene-level inspection will remain descriptive and program-specific.

The analytical unit remains the frozen consensus-program × CRISPR-gene
association. Genes will not be promoted or excluded through a composite score,
manual biological preference, pathway annotation, multi-program recurrence, or
post-hoc thresholding.

For each primary stronger-dependency association, gene-level interpretation
will retain the following evidence dimensions separately:

- lineage-adjusted effect size;
- global FDR-adjusted significance;
- CRISPR model coverage;
- within-lineage directional consistency;
- between-lineage effect heterogeneity;
- pooled-versus-adjusted directional sensitivity;
- recurrence across consensus programs, when present.

Gene names or known biological functions will not determine analytical
eligibility. Biological annotation, if subsequently added, will be treated as
interpretive context only and will not modify the statistical evidence tier.

Accordingly, individual associations may be described as candidate putative
functional vulnerability signals when their primary effect direction is
compatible with stronger dependency. This terminology does not imply causal
dependency, target validation, therapeutic efficacy, or independent
cross-dataset replication.

In [50]:
# =============================================================================
# Freeze descriptive ordering for gene-level interpretation
# =============================================================================

crispr_putative_vulnerability_table = (
    crispr_putative_vulnerability_associations
    .sort_values(
        by=[
            "fdr_q_value",
            "beta",
            "source_gene_label",
        ],
        ascending=[
            True,
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

In [51]:
# =============================================================================
# Construct descriptive putative vulnerability evidence table
# =============================================================================

crispr_putative_vulnerability_table = (
    crispr_putative_vulnerability_table[
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "n_models",
            "beta",
            "standard_error",
            "p_value",
            "fdr_q_value",
            "absolute_beta",
            "n_evaluable_lineages",
            "n_direction_consistent",
            "direction_consistency_fraction",
            "median_lineage_beta",
            "lineage_beta_sd",
            "lineage_beta_iqr",
            "lineage_beta_range",
            "pooled_spearman_rho",
            "lineage_sensitive_direction_reversal",
            "n_significant_programs_for_gene",
        ]
    ]
    .copy()
)

In [52]:
# =============================================================================
# Preview descriptively ordered putative vulnerability associations
# =============================================================================

PUTATIVE_VULNERABILITY_PREVIEW_N = 20

crispr_putative_vulnerability_table.head(
    PUTATIVE_VULNERABILITY_PREVIEW_N
)

,consensus_program_id,source_gene_label,gene_symbol,entrez_id,n_models,beta,standard_error,p_value,fdr_q_value,absolute_beta,n_evaluable_lineages,n_direction_consistent,direction_consistency_fraction,median_lineage_beta,lineage_beta_sd,lineage_beta_iqr,lineage_beta_range,pooled_spearman_rho,lineage_sensitive_direction_reversal,n_significant_programs_for_gene
0,CONSENSUS_TX_02,CDS2 (8760),CDS2,8760,538,-0.105367,0.013888,3.275761e-14,3.381568e-10,0.105367,13,9,0.692308,-0.054281,0.279633,0.211262,1.069097,-0.487514,False,3
1,CONSENSUS_TX_03,CHMP4B (128866),CHMP4B,128866,538,-0.250815,0.036093,3.673351e-12,2.370000e-08,0.250815,13,12,0.923077,-0.287520,0.228449,0.117409,1.035373,-0.536831,False,2
2,CONSENSUS_TX_01,EBF1 (1879),EBF1,1879,538,-0.335312,0.049199,9.397496e-12,4.409561e-08,0.335312,13,9,0.692308,-0.031962,0.188436,0.118488,0.760811,-0.262207,False,1
3,CONSENSUS_TX_02,ELMO2 (63916),ELMO2,63916,538,-0.113738,0.017282,4.666418e-11,1.605715e-07,0.113738,13,9,0.692308,-0.060138,0.194074,0.141427,0.730187,-0.361000,False,2
4,CONSENSUS_TX_03,PRKAR1A (5573),PRKAR1A,5573,538,-0.121383,0.018680,8.128686e-11,2.622263e-07,0.121383,13,12,0.923077,-0.129022,0.099406,0.093303,0.362003,-0.330757,False,2
5,CONSENSUS_TX_03,ITGAV (3685),ITGAV,3685,538,-0.185306,0.029061,1.813057e-10,4.925313e-07,0.185306,13,10,0.769231,-0.113906,0.194411,0.105518,0.713889,-0.472135,False,2
6,CONSENSUS_TX_03,FERMT2 (10979),FERMT2,10979,538,-0.188542,0.029880,2.792370e-10,7.206409e-07,0.188542,13,13,1.000000,-0.101433,0.111823,0.152181,0.344524,-0.570181,False,2
7,CONSENSUS_TX_03,JUN (3725),JUN,3725,538,-0.136004,0.021743,3.970113e-10,9.757970e-07,0.136004,13,13,1.000000,-0.109691,0.101275,0.108092,0.364446,-0.376452,False,2
8,CONSENSUS_TX_03,ELMO2 (63916),ELMO2,63916,538,-0.158995,0.025847,7.676568e-10,1.722722e-06,0.158995,13,12,0.923077,-0.148344,0.143704,0.085000,0.596306,-0.463741,False,2
9,CONSENSUS_TX_02,PSMB5 (5693),PSMB5,5693,538,-0.207240,0.033825,8.962275e-10,1.850351e-06,0.207240,13,12,0.923077,-0.413865,0.660720,0.791019,2.364232,-0.163086,False,1


## Descriptive preview of gene-level associations

The first 20 rows shown above are a deterministic presentation subset of the
492 stronger-dependency associations, ordered by primary global FDR and then by
primary effect direction and source gene label.

This preview is not a biological ranking, candidate-selection threshold, or
validation tier.

The displayed associations illustrate that statistical support, primary effect
magnitude, cross-lineage directional consistency, and between-lineage
heterogeneity do not necessarily vary together. Associations with strong global
FDR support may still show substantial lineage-to-lineage effect dispersion,
whereas other associations show more uniform within-lineage behavior.

Accordingly, no individual gene is promoted or excluded on the basis of this
preview. The complete 492-association table remains the authoritative
putative-vulnerability-oriented result set for notebook 500.

In [53]:
# =============================================================================
# Output directories
# =============================================================================

DEPENDENCY_OUTPUT_DIR = Paths.dependencies
FUNCTIONAL_VULNERABILITY_OUTPUT_DIR = (
    Paths.functional_vulnerabilities
)

In [54]:
# =============================================================================
# Output artifact paths
# =============================================================================

CRISPR_MODEL_COHORT_PATH = (
    DEPENDENCY_OUTPUT_DIR
    / "500_crispr_model_cohort.csv"
)

CRISPR_GENE_COVERAGE_PATH = (
    DEPENDENCY_OUTPUT_DIR
    / "500_crispr_gene_coverage.csv"
)

CRISPR_PRIMARY_ASSOCIATIONS_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_primary_associations.csv"
)

CRISPR_SIGNIFICANT_EVIDENCE_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_primary_significant_evidence.csv"
)

CRISPR_WITHIN_LINEAGE_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_within_lineage_associations.csv"
)

CRISPR_PUTATIVE_VULNERABILITY_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_putative_vulnerability_associations.csv"
)

CRISPR_METADATA_PATH = (
    FUNCTIONAL_VULNERABILITY_OUTPUT_DIR
    / "500_crispr_analysis_metadata.json"
)

In [55]:
# =============================================================================
# Construct CRISPR shared-model cohort artifact
# =============================================================================

crispr_model_cohort = (
    shared_consensus_analysis
    .reset_index()
    [
        [
            "ModelID",
            "OncotreeLineage",
            *consensus_program_ids,
        ]
    ]
    .copy()
)

In [56]:
# =============================================================================
# Construct CRISPR gene-coverage artifact
# =============================================================================

crispr_gene_coverage_artifact = (
    crispr_gene_coverage[
        [
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "n_observed",
            "n_missing",
            "missing_fraction",
            "gene_effect_sd",
            "n_unique",
            "analysis_eligible",
        ]
    ]
    .copy()
)

In [57]:
# =============================================================================
# Construct primary CRISPR association artifact
# =============================================================================

crispr_primary_association_artifact = (
    primary_crispr_associations
    .merge(
        crispr_gene_annotations[
            [
                "source_gene_label",
                "gene_symbol",
                "entrez_id",
            ]
        ],
        on="source_gene_label",
        how="left",
        validate="many_to_one",
    )
    [
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "n_models",
            "beta",
            "standard_error",
            "p_value",
            "fdr_q_value",
            "fdr_significant",
        ]
    ]
    .copy()
)

In [58]:
# =============================================================================
# Construct significant CRISPR evidence artifact
# =============================================================================

crispr_significant_evidence_artifact = (
    crispr_primary_evidence_table
    .copy()
)

In [59]:
# =============================================================================
# Construct within-lineage CRISPR association artifact
# =============================================================================

crispr_within_lineage_artifact = (
    within_lineage_associations
    .merge(
        crispr_gene_annotations[
            [
                "source_gene_label",
                "gene_symbol",
                "entrez_id",
            ]
        ],
        on="source_gene_label",
        how="left",
        validate="many_to_one",
    )
    [
        [
            "consensus_program_id",
            "source_gene_label",
            "gene_symbol",
            "entrez_id",
            "OncotreeLineage",
            "n_models",
            "beta",
        ]
    ]
    .copy()
)

In [60]:
# =============================================================================
# Construct putative vulnerability CRISPR artifact
# =============================================================================

crispr_putative_vulnerability_artifact = (
    crispr_putative_vulnerability_table
    .copy()
)

In [ ]:
# =============================================================================
# Construct CRISPR analysis metadata
# =============================================================================

crispr_analysis_metadata = {
    "notebook": "500_crispr_associations",
    "source_dataset": "DepMap Public 24Q4",
    "source_file": "CRISPRGeneEffect.csv",
    "frozen_consensus_programs": consensus_program_ids,
    "frozen_consensus_model_count": len(consensus_cellline_scores),
    "crispr_source_model_count": len(crispr_gene_effect),
    "shared_model_count": len(shared_consensus_analysis),
    "observed_lineage_count": len(shared_lineage_counts),
    "min_lineage_n": MIN_LINEAGE_N,
    "supported_lineage_count": len(supported_lineages),
    "crispr_source_gene_count": len(crispr_gene_annotations),
    "min_crispr_model_fraction": MIN_CRISPR_MODEL_FRACTION,
    "min_crispr_models": MIN_CRISPR_MODELS,
    "eligible_crispr_gene_count": len(eligible_crispr_genes),
    "primary_hypothesis_count": PRIMARY_HYPOTHESIS_COUNT,
    "primary_model": "gene_effect ~ consensus_score + lineage_fixed_effects",
    "primary_covariance": PRIMARY_COV_TYPE,
    "multiple_testing_method": PRIMARY_FDR_METHOD,
    "primary_fdr_alpha": PRIMARY_FDR_ALPHA,
    "primary_fdr_significant_count": int(
        primary_crispr_associations["fdr_significant"].sum()
    ),
    "putative_vulnerability_association_count": len(
        crispr_putative_vulnerability_artifact
    ),
    "secondary_characterization": [
        "within_lineage_directional_consistency",
        "cross_lineage_effect_heterogeneity",
        "pooled_spearman_context",
        "multi_program_gene_overlap",
    ],
    "interpretation_boundary": (
        "Computational dependency associations and putative functional "
        "vulnerability evidence; not causal dependency, target validation, "
        "therapeutic efficacy, clinical prediction, or independent replication."
    ),
}

In [62]:
# =============================================================================
# Write analysis-ready CRISPR dependency artifacts
# =============================================================================

crispr_model_cohort.to_csv(
    CRISPR_MODEL_COHORT_PATH,
    index=False,
)

crispr_gene_coverage_artifact.to_csv(
    CRISPR_GENE_COVERAGE_PATH,
    index=False,
)

In [63]:
# =============================================================================
# Write processed CRISPR association artifacts
# =============================================================================

crispr_primary_association_artifact.to_csv(
    CRISPR_PRIMARY_ASSOCIATIONS_PATH,
    index=False,
)

crispr_significant_evidence_artifact.to_csv(
    CRISPR_SIGNIFICANT_EVIDENCE_PATH,
    index=False,
)

crispr_within_lineage_artifact.to_csv(
    CRISPR_WITHIN_LINEAGE_PATH,
    index=False,
)

crispr_putative_vulnerability_artifact.to_csv(
    CRISPR_PUTATIVE_VULNERABILITY_PATH,
    index=False,
)

In [64]:
# =============================================================================
# Write CRISPR analysis metadata
# =============================================================================

with CRISPR_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        crispr_analysis_metadata,
        metadata_file,
        indent=2,
    )

In [65]:
# =============================================================================
# Verify written notebook 500 artifacts
# =============================================================================

csv_artifacts = {
    CRISPR_MODEL_COHORT_PATH: crispr_model_cohort,
    CRISPR_GENE_COVERAGE_PATH: crispr_gene_coverage_artifact,
    CRISPR_PRIMARY_ASSOCIATIONS_PATH: crispr_primary_association_artifact,
    CRISPR_SIGNIFICANT_EVIDENCE_PATH: crispr_significant_evidence_artifact,
    CRISPR_WITHIN_LINEAGE_PATH: crispr_within_lineage_artifact,
    CRISPR_PUTATIVE_VULNERABILITY_PATH: crispr_putative_vulnerability_artifact,
}

artifact_checks = []

for artifact_path, artifact_table in csv_artifacts.items():
    observed_shape = (
        pd.read_csv(artifact_path).shape
        if artifact_path.exists()
        else None
    )

    artifact_checks.append(
        artifact_path.exists()
        and observed_shape == artifact_table.shape
    )

with CRISPR_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as metadata_file:
    written_metadata = json.load(metadata_file)

metadata_check = (
    written_metadata == crispr_analysis_metadata
)

print(
    "CSV artifacts verified:",
    sum(artifact_checks),
    "of",
    len(artifact_checks),
)
print("Metadata verified:", metadata_check)
print(
    "Notebook 500 artifact verification:",
    all(artifact_checks) and metadata_check,
)

CSV artifacts verified: 6 of 6
Metadata verified: True
Notebook 500 artifact verification: True


## Notebook 500 conclusion

Notebook 500 completed the CRISPR functional-vulnerability characterization of
the three frozen Phase 4 consensus transcriptomic programs using DepMap Public
24Q4 gene-effect data.

The frozen 713-model consensus cohort yielded 539 models with CRISPR coverage.
After outcome-independent gene-coverage filtering, 17,205 CRISPR genes entered
the prespecified primary lineage-adjusted association family, corresponding to
51,615 program–gene tests.

Primary inference used lineage fixed effects, HC3 covariance estimation, and
global Benjamini–Hochberg correction across the complete hypothesis family.
A total of 944 associations met `FDR < 0.05`.

Among these, 492 associations had the direction compatible with stronger
functional dependency at higher consensus-program score and were retained as
putative functional vulnerability-oriented associations. These correspond to
463 unique genes and remain program-specific analytical units.

Secondary, result-conditioned characterization assessed within-lineage
directional consistency, between-lineage effect heterogeneity, pooled versus
lineage-adjusted direction, and multi-program gene overlap. These analyses were
used for interpretation only and did not redefine primary significance,
eligibility, multiplicity, or program membership.

The analysis identified substantial program-specific structure. Some
associations showed broad directional consistency across supported lineages,
whereas others exhibited marked lineage-dependent effect heterogeneity.
Accordingly, notebook 500 does not define a single pan-cancer vulnerability
ranking or introduce post-hoc effect-size, consistency, or heterogeneity gates.

No frozen provenance-supported cell-line proliferation covariate was available,
so residual proliferation-related confounding remains an explicit limitation.
Residual technical and biological confounding may also remain despite lineage
adjustment.

The resulting CRISPR associations constitute computational dependency evidence
and candidate putative functional vulnerability signals. They do not establish
causal dependency, validated targets, therapeutic efficacy, clinical
prediction, or independent cross-dataset replication.

Notebook 500 is complete. Downstream integration with RNAi belongs to notebook
501 and the later integrated vulnerability layer, and must not retrospectively
redefine the frozen CRISPR analysis reported here.